<div style="display:flex;align-items:center;justify-content:space-between;border-bottom:2px solid #c8962d;padding-bottom:12px;margin-bottom:20px">
  <div>
    <strong>Universidad Externado de Colombia</strong><br>
    <span>Programa de Ciencia de Datos · Machine Learning II</span><br>
    <span>Docente: Wilmer Pineda-Ríos</span>
  </div>
  <img src="../../assets/brand/logo-externado.png" alt="Universidad Externado de Colombia" width="190" onerror="this.style.display='none'">
</div>

# Semana 1 — Baselines y métricas de evaluación

**Objetivo.** Construir referencias simples y elegir métricas coherentes con los costos de error del problema.

## 1. Contexto del caso

Una entidad financiera quiere anticipar que clientes podrian caer en default. Antes de usar modelos sofisticados, necesitamos una pregunta honesta: **un modelo de machine learning mejora una regla simple?**

En esta semana nos concentramos en el baseline con `DummyClassifier` y en la lectura de metricas. El objetivo no es maximizar un numero, sino entender que significa equivocarse en este problema.

## 2. Objetivo de aprendizaje

- Traducir un problema de negocio a una tarea de clasificacion.
- Construir baselines con `DummyClassifier`.
- Leer matriz de confusion, accuracy, precision, recall y F1.
- Entender por que accuracy puede ser enganosa con clases desbalanceadas.
- Redactar una conclusion ejecutiva sin exagerar la evidencia.

## 3. Del negocio a lo tecnico

Pregunta de negocio: **a que clientes deberiamos revisar o acompanar antes de que entren en default?**

Traduccion tecnica:

- Tarea: clasificacion binaria.
- Unidad de analisis: cliente.
- Variable objetivo: `Default`, donde 1 indica default y 0 no default.
- Clase positiva: default.
- Error critico: un falso negativo puede ser costoso porque deja pasar un cliente riesgoso.
- Baseline: una regla ingenua implementada con `DummyClassifier`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

RANDOM_STATE = 20230715

## 4. Datos

Este notebook usa el archivo `Default_Clientes.csv`, ubicado en `datasets/public/` dentro del repositorio del curso.

In [2]:
data_path = Path("../../datasets/public/Default_Clientes.csv")
if not data_path.exists():
    data_path = Path("datasets/public/Default_Clientes.csv")
if not data_path.exists():
    raise FileNotFoundError("No encontre datasets/public/Default_Clientes.csv")

df = pd.read_csv(data_path, sep=";")
df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,Default
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [3]:
df.shape, df.columns.tolist()

((30000, 25),
 ['ID',
  'LIMIT_BAL',
  'SEX',
  'EDUCATION',
  'MARRIAGE',
  'AGE',
  'PAY_0',
  'PAY_2',
  'PAY_3',
  'PAY_4',
  'PAY_5',
  'PAY_6',
  'BILL_AMT1',
  'BILL_AMT2',
  'BILL_AMT3',
  'BILL_AMT4',
  'BILL_AMT5',
  'BILL_AMT6',
  'PAY_AMT1',
  'PAY_AMT2',
  'PAY_AMT3',
  'PAY_AMT4',
  'PAY_AMT5',
  'PAY_AMT6',
  'Default'])

### Diccionario minimo

- `LIMIT_BAL`: monto del credito otorgado.
- `SEX`, `EDUCATION`, `MARRIAGE`: variables categoricas del cliente.
- `AGE`: edad.
- `PAY_0`, `PAY_2`, ..., `PAY_6`: historial de pago mensual.
- `BILL_AMT1`, ..., `BILL_AMT6`: valores facturados.
- `PAY_AMT1`, ..., `PAY_AMT6`: pagos realizados.
- `Default`: variable objetivo.

Riesgo metodologico para discutir: variables medidas despues del momento de decision podrian generar fuga de informacion. En este notebook asumimos que todas las variables estan disponibles antes de decidir.

In [4]:
target = "Default"

class_balance = (
    df[target]
    .value_counts(normalize=True)
    .rename_axis("clase")
    .reset_index(name="proporcion")
)
class_balance

,clase,proporcion
0,0,0.7788
1,1,0.2212


## 5. Particion de datos

La particion debe hacerse antes de entrenar cualquier transformacion. Usamos `stratify` para conservar la proporcion de default/no default en entrenamiento y prueba.

In [5]:
X = df.drop(columns=["ID", target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

y_train.value_counts(normalize=True), y_test.value_counts(normalize=True)

(Default
 0    0.778792
 1    0.221208
 Name: proportion, dtype: float64,
 Default
 0    0.778833
 1    0.221167
 Name: proportion, dtype: float64)

## 6. Baselines con DummyClassifier

`DummyClassifier` no aprende patrones utiles de las variables. Sirve para responder: **que tan bien lo haria una regla ingenua?**

Probamos tres estrategias:

- `most_frequent`: siempre predice la clase mayoritaria.
- `stratified`: predice respetando la distribucion observada de clases.
- `constant`: siempre predice default; puede ser util para entender el extremo de recall maximo.

In [6]:
def evaluate_classifier(name, estimator, X_test, y_test):
    y_pred = estimator.predict(X_test)
    metrics = {
        "modelo": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_default": precision_score(y_test, y_pred, zero_division=0),
        "recall_default": recall_score(y_test, y_pred, zero_division=0),
        "f1_default": f1_score(y_test, y_pred, zero_division=0),
    }
    if hasattr(estimator, "predict_proba"):
        proba = estimator.predict_proba(X_test)[:, 1]
        metrics["roc_auc"] = roc_auc_score(y_test, proba)
    return metrics

dummy_models = {
    "dummy_most_frequent": DummyClassifier(strategy="most_frequent"),
    "dummy_stratified": DummyClassifier(strategy="stratified", random_state=RANDOM_STATE),
    "dummy_all_default": DummyClassifier(strategy="constant", constant=1),
}

baseline_results = []
for name, clf in dummy_models.items():
    clf.fit(X_train, y_train)
    baseline_results.append(evaluate_classifier(name, clf, X_test, y_test))

pd.DataFrame(baseline_results).sort_values("f1_default", ascending=False)

,modelo,accuracy,precision_default,recall_default,f1_default,roc_auc
2,dummy_all_default,0.221167,0.221167,1.000000,0.362222,0.500000
1,dummy_stratified,0.646833,0.204036,0.205727,0.204878,0.488911
0,dummy_most_frequent,0.778833,0.000000,0.000000,0.000000,0.500000


## 7. Que significan las metricas?

Para la clase positiva `Default = 1`:

- **Accuracy**: proporcion total de predicciones correctas. Puede verse bien aunque el modelo ignore la clase minoritaria.
- **Precision**: de los clientes marcados como default, cuantos realmente eran default.
- **Recall** o sensibilidad: de los clientes que realmente eran default, cuantos detectamos.
- **F1**: promedio armonico entre precision y recall. Penaliza cuando una de las dos es muy baja.
- **ROC AUC**: capacidad de ordenar casos positivos por encima de negativos usando probabilidades.

Para regresion, que aparecera en otros problemas:

- **MAE**: error absoluto promedio, en unidades de la variable objetivo.
- **RMSE**: raiz del error cuadratico medio; castiga mas los errores grandes.
- **R2**: proporcion de variabilidad explicada frente a predecir el promedio.

In [7]:
selected_baseline = dummy_models["dummy_most_frequent"]
baseline_pred = selected_baseline.predict(X_test)

cm = confusion_matrix(y_test, baseline_pred)
cm

array([[4673,    0],
       [1327,    0]], dtype=int64)

In [8]:
pd.DataFrame(
    confusion_matrix(y_test, baseline_pred),
    index=["Real no default", "Real default"],
    columns=["Predice no default", "Predice default"],
)

,Predice no default,Predice default
Real no default,4673,0
Real default,1327,0


Interpretacion esperada: el baseline de clase mayoritaria puede tener accuracy aparentemente aceptable, pero recall de default igual a cero. Eso significa que no detecta ningun cliente riesgoso.

## 8. Lectura del diagnóstico técnico

En esta primera semana no buscamos entrenar el mejor modelo. Buscamos diagnosticar si el grupo entiende el punto de partida de cualquier flujo de modelamiento:

- cuál es la clase positiva;
- por qué separamos entrenamiento y prueba;
- qué hace un baseline;
- por qué accuracy puede engañar;
- qué implican precision, recall y F1 para una decisión real.


## 9. De lo técnico al negocio

Complete:

- ¿Qué decisión quiere apoyar la entidad financiera?
- ¿Cuál es la clase positiva y por qué?
- ¿Qué métrica debería mirar primero la entidad financiera?
- ¿Qué muestra el baseline de clase mayoritaria?
- ¿Qué advertencia metodológica debe comunicarse antes de prometer valor?


## 10. Mensaje vendedor

Escriba un resumen ejecutivo de 5 a 8 líneas. Debe incluir:

- baseline usado;
- métrica principal;
- lectura de la matriz de confusión;
- recomendación sobre qué debería hacerse antes de construir modelos más complejos;
- limitación o riesgo.

Ejemplo de estructura:

> Una regla ingenua que siempre predice la clase mayoritaria puede verse aceptable en accuracy, pero no detecta clientes en default. Esto muestra que la evaluaci?n del problema debe priorizar m?tricas asociadas a la clase positiva, especialmente recall y F1. Antes de prometer valor con un modelo complejo, recomendamos acordar el costo de falsos negativos y falsos positivos con el equipo de riesgo.


## 11. Reto del estudiante

Compare los tres baselines (`most_frequent`, `stratified`, `constant`) y defienda cuál sirve mejor como punto de referencia para iniciar el proyecto. No busque el número más alto: explique qué nos enseña cada baseline sobre el problema.
